# Import

In [29]:
import models.juanchitocnn
import models.visiontransformer
import models.efficientnetb0
import data_loader
import torch
from FGSM_attack import FGSMAttacker

# Settings and Hyperparameters

In [25]:
select_model = 'EfficientNet' # ['EfficientNet', 'VisionTransformer', 'JuanchitoCNN']
train = False # Re-train model?

# Hyperparameters
epochs = 5
learning_rate = 0.001
batch_size = 16
weight_decay = 0.01
attack_style='SaltAndPepper'
train_test_split_ratio = 0.8
fgsm_epsilon = 0.01
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Only need to run cells below

## 1. Selecting Model

In [11]:
if select_model == 'EfficientNet':
    project_model = models.efficientnetb0.ProjectEfficientNet(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size)
    print(f'Using model {select_model}')
elif select_model == 'VisionTransformer':
    project_model = models.visiontransformer.ProjectVisionTransformer(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size, weight_decay=weight_decay)
    print(f'Using model {select_model}')
elif select_model == 'JuanchitoCNN':
    project_model = models.juanchitocnn.ProjectJuanchitoCNN(epochs=epochs, learning_rate=learning_rate, batch_size=batch_size)
    print(f'Using model {select_model}')
else:
    print('No valid model selected')



Using model EfficientNet


## 2. Train or Load Model

In [14]:
if train == True:
    regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path='data/train.csv', 
                                                                                       image_folder='data/train_data', 
                                                                                       image_size=(224, 224), 
                                                                                       split_ratio=train_test_split_ratio, 
                                                                                       train_batch_size=batch_size, 
                                                                                       test_batch_size=batch_size)

    project_model.data_load(train_loader, test_loader)
    project_model.train()
elif train == False:
    project_model.model_load()

## 3. Preparing Data (Regular, Augmented, and Attack)

In [32]:
regular_train_loader, regular_test_loader = data_loader.data_to_train_test_dataloaders(csv_path='data/train.csv', 
                                                                                       image_folder='data/train_data', 
                                                                                       image_size=(224, 224), 
                                                                                       split_ratio=train_test_split_ratio, 
                                                                                       train_batch_size=batch_size, 
                                                                                       test_batch_size=batch_size)

aug_train_loader, aug_test_loader = data_loader.data_to_aug_dataloaders(csv_path='data/train.csv', 
                                                                        image_folder='data/train_data', 
                                                                        image_size=(224, 224), 
                                                                        split_ratio=train_test_split_ratio, 
                                                                        train_batch_size=batch_size, 
                                                                        test_batch_size=batch_size)

attack_train_loader, attack_test_loader = data_loader.data_to_attack_dataloaders(csv_path='data/train.csv', 
                                                                                 image_folder='data/train_data', 
                                                                                 image_size=(224, 224), 
                                                                                 split_ratio=train_test_split_ratio, 
                                                                                 train_batch_size=batch_size, 
                                                                                 test_batch_size=batch_size, 
                                                                                 attack_style=attack_style)

fgsm_train_loader, fgsm_test_loader = data_loader.data_to_attack_dataloaders(csv_path='data/train.csv', 
                                                                             image_folder='data/train_data', 
                                                                             image_size=(224, 224), 
                                                                             split_ratio=train_test_split_ratio, 
                                                                             train_batch_size=1, 
                                                                             test_batch_size=1, 
                                                                             attack_style=attack_style)

## 4. Evaluate Regular Test Data

In [18]:
project_model.data_load(regular_train_loader, regular_test_loader)
regular_correct, total, regular_incorrect_preds = project_model.evaluate()

Test Accuracy: 99.66%


## 5. Evaluate Augmented Test Data

In [19]:
project_model.data_load(None, aug_test_loader)
aug_correct, total, aug_incorrect_preds = project_model.evaluate()

Test Accuracy: 99.67%


## 6. Evaluate Attack Test Data

In [20]:
project_model.data_load(None, attack_test_loader)
attack_correct, total, attack_incorrect_preds = project_model.evaluate()

Test Accuracy: 87.86%


## 7. FGSM

In [ ]:
attacker = FGSMAttacker(project_model.model, device)
acc, adv_examples = attacker.attack(fgsm_test_loader, fgsm_epsilon)